In [ ]:
bronze_path = "/content/bronze"
df_bronze = spark.read.format("delta").load(bronze_path)

df_silver = df_bronze.drop("udi", "product_id")

df_silver = df_silver.na.drop()  # or use .fill() if you prefer imputing

from pyspark.sql.functions import col, sum

df_silver.select([sum(col(c).isNull().cast("int")).alias(c) for c in df_silver.columns]).show()


In [ ]:
df_silver.columns


In [ ]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder
from pyspark.ml import Pipeline

indexer = StringIndexer(inputCol="type", outputCol="type_index")
encoder = OneHotEncoder(inputCol="type_index", outputCol="type_ohe")

pipeline = Pipeline(stages=[indexer, encoder])
df_silver = pipeline.fit(df_silver).transform(df_silver).drop("type", "type_index")

silver_path = "/content/silver"
df_silver.write.format("delta").mode("overwrite").save(silver_path)

df_silver_read = spark.read.format("delta").load(silver_path)
df_silver_read.show(5)



In [ ]:
silver_path = "/content/silver"
df_silver.write.format("delta").mode("overwrite").save(silver_path)
